PHASE 8: CIRCUIT MAP BUILDER

Draw each circuit outline Phase 1 telemetry, overlays DRS zones, braking zones, sector boundaries, and pit lane markers, and save each as an interactive Plotly JSON file

In [1]:
import pandas as pd
import numpy as np
import os
import logging
import warnings
import json

import fastf1
import plotly.graph_objects as go

warnings.filterwarnings('ignore')

SETUP

In [2]:
BASE = r'C:\Users\adity\Desktop\BoxBox'

os.makedirs(os.path.join(BASE, 'circuit_maps'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'data', 'outputs'), exist_ok=True)
os.makedirs(os.path.join(BASE, 'cache'), exist_ok=True)

logger_name = __name__  
log = logging.getLogger(logger_name)
if log.hasHandlers():
    log.handlers.clear()

log.setLevel(logging.INFO)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(
    r'C:\Users\adity\Desktop\BoxBox\data\outputs\phase8_log.txt'
)
file_handler.setFormatter(formatter)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)

log.addHandler(file_handler)
log.addHandler(stream_handler)

fastf1.Cache.enable_cache(os.path.join(BASE, 'cache'))

RACE CALENDAR 2024

Reused from Phase 1 - Brazil will be excluded as per our documented data coverage note (wet-weather race, no dry compound data, out of scope for this project)

In [3]:
CALENDAR_2024 = [
    {'name': 'Bahrain', 'round': 1}, {'name':'Saudi Arabia', 'round': 2},
    {'name': 'Australia', 'round': 3}, {'name': 'Japan', 'round': 4},
    {'name': 'China', 'round': 5}, {'name': 'Miami', 'round': 6},
    {'name': 'Emilia Romagna', 'round': 7}, {'name': 'Monaco', 'round': 8},
    {'name': 'Canada', 'round': 9}, {'name': 'Spain', 'round': 10},
    {'name': 'Austria', 'round': 11}, {'name': 'Great Britain', 'round': 12},
    {'name': 'Hungary', 'round': 13}, {'name': 'Belgium', 'round': 14},
    {'name': 'Netherlands', 'round': 15}, {'name': 'Italy', 'round': 16},
    {'name': 'Azerbaijan', 'round': 17}, {'name': 'Singapore', 'round': 18},
    {'name': 'United States', 'round': 19}, {'name': 'Mexico', 'round': 20},
    {'name': 'Las Vegas', 'round': 22}, {'name': 'Qatar', 'round': 23},
    {'name': 'Abu Dhabi', 'round': 24}
]

LOAD TELEMETRY FOR ONE CIRCUIT

Re-loads Qualifying sessoin telemetry with channels (Speed, DRS, Distance) needed for zone detection - Phase 1 only kept X/Y/Z Status, so we pull the richer telemetry here

In [4]:
def load_circuit_telemetry(round_number, year=2024):
    try:
        session = fastf1.get_session(year, round_number, 'Q')
        session.load(laps=True, telemetry=True, weather = False, messages=False)

        fastest_lap = session.laps.pick_fastest()
        if fastest_lap is None:
            return None, None

        telemetry = fastest_lap.get_telemetry()
        return telemetry, fastest_lap
    except Exception as e:
        log.error(f" Failed to load telemetry data for round {round_number}: {e}")
        return None, None

DETECT BRAKING ZONES

A breaking zone is the reigon where speed drops sharply between consecutive telemetry points. We flag the top percentile of speed decreases as "heavy braking"

In [5]:
def detect_braking_zones(telemetry, threshold_percentile=90, expand_points=3):
    telemetry = telemetry.copy()
    telemetry['SpeedDelta'] = telemetry['Speed'].diff()

    braking_deltas = telemetry[telemetry['SpeedDelta'] < 0]['SpeedDelta']

    if braking_deltas.empty:
        telemetry['IsBraking'] = False
        return telemetry

    threshold = np.percentile(braking_deltas, 100 - threshold_percentile)
    initial_flag = telemetry['SpeedDelta'] <= threshold

    # Expand each flagged point outward by a few neighboring
    # telemetry samples on both sides. Braking events are often
    # only 1-2 raw points wide, which renders as an invisible
    # dot — widening them into a short connected run makes them
    # render as a solid segment, matching how DRS zones look.
    expanded = initial_flag.copy()
    flagged_idx = np.where(initial_flag.values)[0]
    for i in flagged_idx:
        lo = max(0, i - expand_points)
        hi = min(len(expanded), i + expand_points + 1)
        expanded.iloc[lo:hi] = True

    telemetry['IsBraking'] = expanded
    return telemetry

DETECT DRS ZONES

Uses FastF1's DRS telemetry channel directly. DRS values of 10, 12, 14 indicate the system was active (per FastF1 docs): other values means available-but-not-active or unavailable

In [6]:
def detect_drs_zones(telemetry):
    telemetry = telemetry.copy()
    if 'DRS' in telemetry.columns:
        telemetry['IsDRSActive'] = telemetry['DRS'].isin([10, 12, 14])
    else:
        telemetry['IsDRSActive'] = False
    return telemetry

APPROXIMATE SECTOR BOUNDARIES

Uses cumulative Distance and the fastest lap's sector times to estimate where along the track each sector transition occurs, proportionally.

In [7]:
def approximate_sector_boundaries(telemetry, fastest_lap):
    total_distance = telemetry['Distance'].max()

    try:
        s1_time = fastest_lap['Sector1Time'].total_seconds()
        s2_time = fastest_lap['Sector2Time'].total_seconds()
        s3_time = fastest_lap['Sector3Time'].total_seconds()
        total_time = s1_time + s2_time +s3_time

        '''Approximate sector boundary distance proportionally
        to sector time share (a simplification, since pace
        varies by sector, but reasonable for a reference map)'''
        s1_boundary_dist = total_distance * (s1_time / total_time)
        s2_boundary_dist = total_distance * ((s1_time + s2_time) / total_time)

        return s1_boundary_dist, s2_boundary_dist
    except Exception:
        return total_distance / 3, total_distance * 2/3

BUILD THE PLOTLY CIRCUIT MAP

In [8]:
import plotly.graph_objects as go
import numpy as np
from scipy.ndimage import uniform_filter1d

def smooth_track(x, y, window=15):
    x_smooth = uniform_filter1d(x, size=window, mode='wrap')
    y_smooth = uniform_filter1d(y, size=window, mode='wrap')
    return x_smooth, y_smooth


def compute_offset_line(x, y, offset_dist=25, smooth_window=9):
    dx = np.gradient(x)
    dy = np.gradient(y)
    length = np.sqrt(dx**2 + dy**2)
    length[length == 0] = 1
    nx = -dy / length
    ny = dx / length
    x_off = x + nx * offset_dist
    y_off = y + ny * offset_dist
    # extra smoothing pass on the offset line itself — this is
    # what removes the jittery/wobbly look
    x_off = uniform_filter1d(x_off, size=smooth_window, mode='wrap')
    y_off = uniform_filter1d(y_off, size=smooth_window, mode='wrap')
    return x_off, y_off


def detect_turn_markers(x, y, min_gap=25, angle_threshold=15):
    dx = np.gradient(x)
    dy = np.gradient(y)
    heading = np.arctan2(dy, dx)
    heading_change = np.abs(np.gradient(np.unwrap(heading)))

    turn_idxs = []
    last_idx = -min_gap
    for i in range(len(heading_change)):
        if heading_change[i] > np.radians(angle_threshold):
            if i - last_idx >= min_gap:
                turn_idxs.append(i)
                last_idx = i
    return turn_idxs


def build_circuit_map(circuit_name, telemetry, s1_dist=None, s2_dist=None):
    fig = go.Figure()

    DARK_BG = '#0D0D0D'
    TRACK_COLOR = '#5A5A5A'
    TRACK_INNER = '#8A8A8A'
    BRAKE_COLOR = '#E8002D'
    DRS_COLOR = '#00E676'
    SPEEDTRAP_COLOR = '#E91E63'
    STARTFINISH_COLOR = '#FFFFFF'

    x_raw = telemetry['X'].values
    y_raw = telemetry['Y'].values
    x, y = smooth_track(x_raw, y_raw, window=15)

    # ── TRACK OUTLINE ──────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines',
        line=dict(color=TRACK_COLOR, width=12),
        hoverinfo='skip', showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=x, y=y, mode='lines',
        line=dict(color=TRACK_INNER, width=5),
        hoverinfo='skip', showlegend=False
    ))

    # ── BRAKING ZONES — solid red, on track ───────────────────
    def add_solid_zone(mask_col, color, width=7):
        mask = telemetry[mask_col].values
        n = len(mask)
        i = 0
        while i < n:
            if mask[i]:
                j = i
                while j < n and mask[j]:
                    j += 1
                if j - i < 2:
                    j = min(n, i + 2)
                fig.add_trace(go.Scatter(
                    x=x[i:j], y=y[i:j], mode='lines',
                    line=dict(color=color, width=width),
                    showlegend=False,
                    hovertemplate='Heavy Braking Zone<extra></extra>'
                ))
                i = j
            else:
                i += 1

    add_solid_zone('IsBraking', BRAKE_COLOR)

    # ── DRS ZONES — smoothed dashed line, offset outside track ─
    drs_mask = telemetry['IsDRSActive'].values
    x_off, y_off = compute_offset_line(x, y, offset_dist=20, smooth_window=9)

    n = len(drs_mask)
    i = 0
    while i < n:
        if drs_mask[i]:
            j = i
            while j < n and drs_mask[j]:
                j += 1
            fig.add_trace(go.Scatter(
                x=x_off[i:j], y=y_off[i:j], mode='lines',
                line=dict(color=DRS_COLOR, width=3, dash='dot'),
                showlegend=False,
                hovertemplate='DRS Detection Zone<extra></extra>'
            ))
            i = j
        else:
            i += 1

    # ── TURN NUMBER MARKERS ────────────────────────────────────
    turn_idxs = detect_turn_markers(x, y, min_gap=30, angle_threshold=12)
    if turn_idxs:
        fig.add_trace(go.Scatter(
            x=[x[i] for i in turn_idxs],
            y=[y[i] for i in turn_idxs],
            mode='markers+text',
            marker=dict(color='#1A1A1A', size=20, line=dict(color='#FFFFFF', width=2)),
            text=[str(n + 1) for n in range(len(turn_idxs))],
            textfont=dict(color='#FFFFFF', size=9, family='Arial Black'),
            textposition='middle center',
            showlegend=False,
            hoverinfo='skip'
        ))

    # ── SPEED TRAP — marker only, no on-map text ───────────────
    if 'Speed' in telemetry.columns:
        max_speed_idx = telemetry['Speed'].values.argmax()
        fig.add_trace(go.Scatter(
            x=[x[max_speed_idx]], y=[y[max_speed_idx]],
            mode='markers',
            marker=dict(color=SPEEDTRAP_COLOR, size=11, symbol='square',
                        line=dict(color='#000000', width=1)),
            showlegend=False,
            hovertemplate='Speed Trap<extra></extra>'
        ))

    # ── START / FINISH ──────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=[x[0]], y=[y[0]],
        mode='markers',
        marker=dict(color=STARTFINISH_COLOR, size=11, symbol='square',
                     line=dict(color='#000000', width=1)),
        showlegend=False,
        hovertemplate='Start/Finish<extra></extra>'
    ))

    # ── LEGEND — dummy square-marker traces, side legend ───────
    legend_items = [
        ('Heavy Braking Zone', BRAKE_COLOR),
        ('DRS Detection Zone', DRS_COLOR),
        ('Speed Trap', SPEEDTRAP_COLOR),
        ('Start/Finish', STARTFINISH_COLOR),
    ]
    for label, color in legend_items:
        fig.add_trace(go.Scatter(
            x=[None], y=[None],
            mode='markers',
            marker=dict(color=color, size=11, symbol='square',
                         line=dict(color='#000000', width=1)),
            name=label,
            showlegend=True
        ))

    # ── LAYOUT ───────────────────────────────────────────────────
    fig.update_layout(
        title=dict(text=f"{circuit_name}", font=dict(color='#FFFFFF', size=22), x=0.02),
        plot_bgcolor=DARK_BG,
        paper_bgcolor=DARK_BG,
        font=dict(color='#CCCCCC'),
        xaxis=dict(visible=False, scaleanchor='y', scaleratio=1),
        yaxis=dict(visible=False),
        showlegend=True,
        legend=dict(
            font=dict(color='#CCCCCC', size=11),
            bgcolor='rgba(0,0,0,0)',
            orientation='v',
            yanchor='top', y=1,
            xanchor='left', x=1.02
        ),
        margin=dict(l=10, r=140, t=60, b=10),
        height=650
    )

    return fig

MAIN PIPELINE

In [9]:
def main():
    log.info("BOXBOX - Phase 8: Circuit Map Builder")
    log.info("-" * 50)

    success_count = 0
    failed_circuits = []

    for race in CALENDAR_2024:
        circuit_name = race['name']
        log.info(f"\nBuilding map for {circuit_name}...")

        telemetry, fastest_lap = load_circuit_telemetry(race['round'])

        if telemetry is None:
            log.warning(f" {circuit_name}: No telemetry available, skipping")
            failed_circuits.append(circuit_name)
            continue

        telemetry = detect_braking_zones(telemetry)
        telemetry = detect_drs_zones(telemetry)
        s1_dist, s2_dist = approximate_sector_boundaries(telemetry, fastest_lap)

        fig = build_circuit_map(circuit_name, telemetry, s1_dist, s2_dist)

        #Save as Plotly JSON - loadable directly to Streamlit
        safe_name = circuit_name.lower().replace(' ', '_')
        output_path = os.path.join(BASE, 'circuit_maps', f'{safe_name}.json')
        fig.write_json(output_path)

        n_braking = telemetry['IsBraking'].sum()
        n_drs = telemetry['IsDRSActive'].sum()
        log.info(f" {circuit_name}: saved |"
                f"{n_braking} braking points, {n_drs} DRS points")

        success_count += 1

    log.info(f"\nCircuit maps built: {success_count}/{len(CALENDAR_2024)}")
    if failed_circuits:
        log.warning(f" Failed circuits: {failed_circuits}")
    log.info(f"\nMaps saved to: {os.path.join(BASE, 'circuit_maps')}")

if __name__ =='__main__':
    main()

2026-09-23 11:01:53,725 - INFO - BOXBOX - Phase 8: Circuit Map Builder
2026-09-23 11:01:53,731 - INFO - --------------------------------------------------
2026-09-23 11:01:53,731 - INFO - 
Building map for Bahrain...
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '63', '55', '11', '14', '4', '81', '44', '27', '22', '18', '23', '3', '20', '77', '24', '2', '31', '10']
2026-09-23 11